# PI-BDP × N-CMAPSS — 그림 수정본 (fig3, fig6, fig7, fig8)

Phase 3에 저장된 결과로 **그림 4개만 다시 그립니다.** 새로 평가하거나 학습하지 않습니다. CPU 런타임으로 1~2분이면 끝납니다.

| 그림 | 고친 점 |
|---|---|
| fig3_forest | p값이 0.001 미만이면 `0.000` 대신 `< 0.001`로 표기 |
| fig6_calibration | DS04처럼 보정 전후 포함률이 같은 점이 겹쳐 안 보이던 문제: 보정 전을 속이 빈 큰 원으로 그리고, 겹친 행에 설명 추가 |
| fig7_examples | DS04 보정 구간 위쪽이 그림 밖으로 잘리던 문제: 세 패널의 y축을 따로 잡아 전체 구간이 보이게 함 |
| fig8_dev_vs_test | 회색 `DS04` 표시가 어느 모델 점인지 헷갈리던 문제: 모델 색으로 칠하고 위치를 겹치지 않게 조정 |

**실행:** 런타임 → 모두 실행. 결과는 드라이브 `PI-BDP_NCMAPSS/figures/`의 같은 파일 이름(PNG 300 dpi, PDF)을 덮어쓰고, 마지막에 `figures_fixed.zip`으로 묶습니다. 이전 그림은 `figures/_old/`에 옮겨 둡니다.

In [ ]:
import os, sys, json, shutil
import numpy as np, pandas as pd, matplotlib as mpl, matplotlib.pyplot as plt
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/PI-BDP_NCMAPSS'
else:
    ROOT = os.environ.get('NCM_ROOT', './ncm')
P3, OUT = f'{ROOT}/phase3', f'{ROOT}/figures'
OLD = f'{OUT}/_old'; os.makedirs(OLD, exist_ok=True)
for n in ['fig3_forest', 'fig6_calibration', 'fig7_examples', 'fig8_dev_vs_test']:
    for ext in ('png', 'pdf'):
        f = f'{OUT}/{n}.{ext}'
        if os.path.exists(f) and not os.path.exists(f'{OLD}/{n}.{ext}'): shutil.copy2(f, f'{OLD}/{n}.{ext}')
cfg = json.load(open(f'{ROOT}/phase1/config.json')); SUBSETS = sorted(cfg['k'])
SHORT = {s: s.split('-')[0] for s in SUBSETS}
CAL_EPS = 0.5   # 고정 라이브러리(pibdp_ncm.py)의 CAL_EPS와 같은 값
mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': '#8a8984', 'axes.linewidth': 0.6,
    'axes.grid': True, 'grid.color': '#e6e5e0', 'grid.linewidth': 0.5, 'axes.axisbelow': True,
    'xtick.color': '#52514e', 'ytick.color': '#52514e', 'axes.labelcolor': '#0b0b0b',
    'lines.linewidth': 1.5, 'savefig.dpi': 300, 'figure.dpi': 110})
COL = dict(v0='#2a78d6', gb='#eb6834', lstm='#1baf7a', rf='#eda100', gray='#b4b3ad', ink='#0b0b0b', ink2='#52514e')
def save(fig, name):
    fig.savefig(f'{OUT}/{name}.png', bbox_inches='tight'); fig.savefig(f'{OUT}/{name}.pdf', bbox_inches='tight'); plt.show()
def fmt_p(p):
    return '< 0.001' if p < 0.001 else f'= {p:.3f}'
print('준비 완료. 이전 그림 백업:', OLD)

In [ ]:
# === 그림 3: 비교 검정 forest plot (p값 표기 수정) ===
T = pd.read_csv(f'{P3}/phase3_tests.csv')
lab = {'V0 - RF': 'RF', 'V0 - GB': 'GB', 'V0 - MLP': 'MLP', 'V0 - LSTM': 'LSTM', 'V0 - CNN': 'CNN',
       'V0 - V1': 'PI-BDP V1 (bootstrap)', 'V0 - RawCNN-1Hz': '1 Hz raw CNN (reference)',
       'V0 - PI-noWeibull': 'PI-noWeibull (ablation)', 'V0 - DHI-Exp': 'DHI-Exp (ablation)'}
T = T.iloc[::-1].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(5.4, 3.2))
for i, r in T.iterrows():
    prim = r.family == 'primary'; c = COL['v0'] if prim else COL['ink2']
    ax.plot([r.ci_lo, r.ci_hi], [i, i], color=c, lw=1.6, solid_capstyle='butt')
    ax.plot(r['diff'], i, 'o', ms=5, color=c, mec='white', mew=1.0)
    txt = f"p (Holm) {fmt_p(r.p_holm)}" if prim else f"p {fmt_p(r.p_boot)} (unadj.)"
    ax.text(1.0, i, '  ' + txt, transform=ax.get_yaxis_transform(), va='center', fontsize=6.5, color=COL['ink2'])
ax.axvline(0, color=COL['ink'], lw=0.8)
ax.set_yticks(range(len(T))); ax.set_yticklabels([lab[c] for c in T.comparison])
ax.set_xlabel('RMSE(PI-BDP V0) − RMSE(model)  [cycles]   (negative: V0 more accurate)')
ax.grid(axis='y', visible=False)
fig.tight_layout(); save(fig, 'fig3_forest')

In [ ]:
# === 그림 6: 예측구간 포함률 (겹친 점 표시 수정) ===
PR = pd.read_csv(f'{P3}/phase3_prob.csv'); v = PR[PR.model == 'PI-BDP_V0']
lv = [50, 80, 90, 95]; a = v[v.scope == 'ALL'].iloc[0]
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.9), gridspec_kw=dict(width_ratios=[1, 1.5]))
ax = axs[0]
ax.plot([45, 100], [45, 100], color=COL['ink2'], lw=0.8, ls='--', label='Ideal')
ax.plot(lv, [100 * a[f'raw_cov{l}'] for l in lv], '-o', color=COL['gb'], ms=4, mec='white', label='Raw')
ax.plot(lv, [100 * a[f'cal_cov{l}'] for l in lv], '-o', color=COL['v0'], ms=4, mec='white', label='Calibrated (pre-registered)')
ax.set_xlabel('Nominal coverage (%)'); ax.set_ylabel('Empirical coverage, test (%)'); ax.legend(loc='upper left')
ax.set_title('(a) All test cycles', loc='left')
ax = axs[1]; sv = v[v.scope != 'ALL'].reset_index(drop=True)
for i, r in sv.iterrows():
    x0, x1 = 100 * r.raw_cov90, 100 * r.cal_cov90
    ax.plot([x0, x1], [i, i], color=COL['gray'], lw=1.2, zorder=1)
    ax.plot(x0, i, 'o', ms=8, mfc='none', mec=COL['gb'], mew=1.4, zorder=3)          # raw: 속이 빈 큰 원
    ax.plot(x1, i, 'o', ms=4.5, color=COL['v0'], mec='white', mew=0.6, zorder=4)     # calibrated: 작은 채운 원
    if abs(x1 - x0) < 0.5:
        ax.annotate('raw = calibrated', (x1, i), xytext=(-8, 0), textcoords='offset points',
                    ha='right', va='center', fontsize=6, color=COL['ink2'])
ax.axvline(90, color=COL['ink2'], lw=0.8, ls='--')
ax.set_yticks(range(len(sv))); ax.set_yticklabels([SHORT[s] for s in sv.scope]); ax.invert_yaxis()
ax.set_xlabel('Empirical coverage of the 90% interval (%)')
ax.set_title('(b) By subset: raw (open orange) → calibrated (blue)', loc='left')
ax.grid(axis='y', visible=False)
fig.tight_layout(); save(fig, 'fig6_calibration')

In [ ]:
# === 그림 7: 예시 엔진 (y축을 패널별로, 구간 전체가 보이게) ===
cal = json.load(open(f'{ROOT}/phase1/calibration.json'))['c']; c90 = cal['90']
fig, axs = plt.subplots(1, 3, figsize=(7.2, 2.6))
for k, (ax, s) in enumerate(zip(axs, ['DS02-006', 'DS04', 'DS08a-009'])):
    d = pd.read_csv(f'{P3}/test_PI-BDP_V0_{s}.csv.gz')
    ur = d.groupby('unit').apply(lambda g: np.sqrt(np.mean((g.q500 - g.true) ** 2))).sort_values()
    u = ur.index[(len(ur) - 1) // 2]; g = d[d.unit == u].sort_values('cycle')
    m = g.q500.values
    lo = np.maximum(m - c90 * np.maximum(m - g.q050.values, CAL_EPS), 0)
    hi = m + c90 * np.maximum(g.q950.values - m, CAL_EPS)
    ax.fill_between(g.cycle, g.q050, g.q950, color=COL['v0'], alpha=0.22, lw=0, label='90% interval (raw)')
    ax.plot(g.cycle, lo, color=COL['v0'], lw=0.8, ls=':', label='90% interval (calibrated)'); ax.plot(g.cycle, hi, color=COL['v0'], lw=0.8, ls=':')
    ax.plot(g.cycle, m, color=COL['v0'], lw=1.6, label='Median prediction')
    ax.plot(g.cycle, g.true, color=COL['ink'], lw=1.0, ls='--', label='True RUL')
    top = 1.05 * max(np.nanmax(hi), np.nanmax(g.q950.values), np.nanmax(g.true.values))
    ax.set_ylim(0, top)
    ax.set_title(f'{SHORT[s]} unit {u} (RMSE {ur[u]:.1f})', loc='left'); ax.set_xlabel('Flight cycle')
    print(f'{s} unit {u}: y축 상한 {top:.1f}')
axs[0].set_ylabel('RUL (cycles)'); axs[0].legend(loc='upper right', fontsize=6)
fig.tight_layout(); save(fig, 'fig7_examples')

In [ ]:
# === 그림 8: 서브셋별 dev CV RMSE 대 test RMSE (DS04 표시를 모델 색으로) ===
def per_seed_subset(df):
    return pd.concat([g.groupby('subset').apply(lambda x: np.sqrt(np.mean((x.pred - x.true) ** 2))) for _, g in df.groupby('seed')], axis=1).mean(1)
dev = pd.read_csv(f'{ROOT}/phase2/phase2_cpu_per_subset.csv', index_col=0)
dev_lstm = per_seed_subset(pd.read_csv(f'{ROOT}/phase2/preds_LSTM.csv.gz'))
test = pd.read_csv(f'{P3}/phase3_per_subset.csv', index_col=0)
test_gb = per_seed_subset(pd.read_csv(f'{P3}/test_GB.csv.gz')); test_lstm = per_seed_subset(pd.read_csv(f'{P3}/test_LSTM.csv.gz'))
pairs = [('PI-BDP V0', dev['PI-BDP (V0, primary)'], test['PI-BDP_V0'], COL['v0'], (-6, -11), 'right'),
         ('GB', dev['GB'], test_gb, COL['gb'], (5, 4), 'left'),
         ('LSTM', dev_lstm, test_lstm, COL['lstm'], (5, 4), 'left')]
fig, ax = plt.subplots(figsize=(3.6, 3.4)); lim = (3, 17)
ax.plot(lim, lim, color=COL['ink2'], lw=0.8, ls='--')
i = SUBSETS.index('DS04')
for name, x, y, c, off, ha in pairs:
    x, y = x.reindex(SUBSETS), y.reindex(SUBSETS)
    ax.plot(x, y, 'o', color=c, ms=5, mec='white', mew=0.8, label=name)
    ax.plot(x.iloc[i], y.iloc[i], 'o', ms=8, mfc='none', mec=c, mew=1.0)   # DS04 점 강조
    ax.annotate('DS04', (x.iloc[i], y.iloc[i]), xytext=off, textcoords='offset points', ha=ha,
                fontsize=6.5, color=c, fontweight='bold')
ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect('equal')
ax.set_xlabel('Dev LOUO-CV RMSE (cycles)'); ax.set_ylabel('Test RMSE (cycles)')
ax.legend(loc='upper left', title='Ringed points: DS04', title_fontsize=6.5)
fig.tight_layout(); save(fig, 'fig8_dev_vs_test')

In [ ]:
import zipfile
z = f'{ROOT}/figures_fixed.zip'
with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as f:
    for n in ['fig3_forest', 'fig6_calibration', 'fig7_examples', 'fig8_dev_vs_test']:
        for ext in ('png', 'pdf'): f.write(f'{OUT}/{n}.{ext}', f'{n}.{ext}')
print('완료:', z)